# task1-lora
Fine-tune Qwen2.5-0.5B-Instruct on order requests. The 60 held-out examples cover 15 sentence patterns and four order numbers per pattern. Report exact label accuracy and inspect the actual errors.

Select Runtime → Change runtime type → hosted GPU, then Run all. Use a fresh runtime for each notebook. Save the downloaded results before disconnecting.

In [ ]:
%pip -q install transformers==4.57.1 peft==0.17.1 accelerate==1.10.1

In [ ]:
import json
import random
import subprocess
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "Select a hosted GPU runtime in Colab."
set_seed(42)
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).cuda()
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
print(gpu)

# The same order number can appear in many requests, so split by both wording and number.
train_forms = {
    "CANCEL": ["Cancel order {}.", "Please call off order {}.", "I no longer want order {}.", "Stop order {} before dispatch."],
    "TRACK": ["Where is order {}?", "Tell me the status of order {}.", "Has order {} shipped?", "Find my parcel for order {}."],
    "RETURN": ["Send order {} back.", "I want to return order {}.", "Arrange a pickup for order {}.", "Order {} arrived. I want a refund and return."],
}
test_forms = {
    "CANCEL": ["I haven't received {}. Stop it coming.", "I don't want tracking for {}, just cancel it.", "Please void {}.", "Before you ship {}, take it off my purchases.", "I changed my mind about {} before dispatch."],
    "TRACK": ["Don't cancel {}. Tell me where it is.", "How far away is {}?", "Can you trace {} for me?", "Any delivery update on {}?", "I might return {} later. Has it left yet?"],
    "RETURN": ["I've opened {} and want to send it back.", "I received {}. Can you take it off my hands?", "Don't track {}. Start a return.", "I need the reverse pickup for {}.", "The wrong item arrived in {}. Arrange collection."],
}
def dataset(forms, first, repeats):
    rows = []
    for label, patterns in forms.items():
        for pattern in patterns:
            for _ in range(repeats):
                rows.append((pattern.format(first + len(rows)), label))
    return rows

train = dataset(train_forms, 1000, 10)
test = dataset(test_forms, 8000, 4)
assert len(test) == 60 and len(set(q for q, _ in train + test)) == len(train) + len(test)
def prompt(text):
    return tokenizer.apply_chat_template([
        {"role": "system", "content": "Classify the order request. Reply with exactly one label: CANCEL, TRACK or RETURN."},
        {"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def evaluate(net):
    net.eval()
    tokenizer.padding_side = "left"
    rows = []
    for start in range(0, len(test), 8):
        batch = test[start:start + 8]
        inputs = tokenizer([prompt(q) for q, _ in batch], padding=True, return_tensors="pt").to("cuda")
        ids = net.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outputs = tokenizer.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        for (q, label), output in zip(batch, outputs):
            rows.append({"request": q, "expected": label, "output": output.strip(), "correct": output.strip() == label})
    return rows

base = evaluate(model)
config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, config)
model.print_trainable_parameters()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=0.0002)
rng = random.Random(42)
model.train()
for epoch in range(3):
    rng.shuffle(train)
    for start in range(0, len(train), 4):
        batch = train[start:start + 4]
        sequences, targets = [], []
        for q, label in batch:
            prefix = tokenizer.encode(prompt(q), add_special_tokens=False)
            answer = tokenizer.encode(label, add_special_tokens=False) + [tokenizer.eos_token_id]
            sequences.append(prefix + answer)
            targets.append([-100] * len(prefix) + answer)
        width = max(map(len, sequences))
        mask = [[1] * len(s) + [0] * (width-len(s)) for s in sequences]
        ids = [s + [tokenizer.pad_token_id] * (width-len(s)) for s in sequences]
        labels = [t + [-100] * (width-len(t)) for t in targets]
        loss = model(input_ids=torch.tensor(ids, device="cuda"), attention_mask=torch.tensor(mask, device="cuda"),
                     labels=torch.tensor(labels, device="cuda"), use_cache=False).loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print("Finished epoch", epoch + 1)
tuned = evaluate(model)
result = {"gpu": gpu, "model": model_id, "base": base, "tuned": tuned}
for name, rows in [("base", base), ("tuned", tuned)]:
    hits = sum(row["correct"] for row in rows)
    print(f"{name}: {hits}/{len(rows)} = {hits/len(rows):.1%}")
print("Actual tuned failures:", [row for row in tuned if not row["correct"]])
Path("task1-results.json").write_text(json.dumps(result, indent=2))
from google.colab import files
files.download("task1-results.json")